# Porto Taxi Dataset

This notebook contains test code in regard to the porto taxi dataset [1]. 

[1] L. Moreira-Matias, M. Ferreira, J. Mendes-Moreira, L. L., and J. J., “Porto Taxi - Taxi Service Trajectory - Prediction Challenge, ECML PKDD 2015.” UCI Machine Learning Repository, 2015. doi: 10.24432/C55W25.


In [ ]:
%load_ext autoreload
%autoreload 2

RunningInCOLAB = 'google.colab' in str(get_ipython())
if RunningInCOLAB:
    # Move to default colab folder
    %cd / content
    # Check if repository is already cloned
    import os

    if not os.path.isdir("stg"):
        # Clone repository
        !git clone {config.GITHUB_URL} {config.MODULE_NAME}
    # Change to repository directory
    %cd {config.MODULE_NAME}
    # Only install requirements not already installed by Colab
    # !pip install opacus
    # SLOW: Only execute the following line if you encounter an error regarding a package not being installed
    # !pip install -r requirements.txt
else:
    import sys

    sys.path.append('..')  # Required for finding the `stg` module

In [ ]:
import logging
import multiprocessing as mp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.autonotebook import tqdm

import privtrajgen.datasets.constants
from privtrajgen import config
from privtrajgen.datasets import SPATIAL_COLUMNS, REF_POINT_PORTO, SCALE_FACTOR_PORTO
from privtrajgen.datasets.padding import resample_trajectory
from privtrajgen.utils import normalize_points
from privtrajgen.utils.helpers import printc
from privtrajgen.utils.logger import configure_root_loger

log = configure_root_loger(logging.INFO)

In [ ]:
# Filepaths
PORTO_RAW_FILEPATH = os.path.join(config.BASE_DIR, "data/porto_raw/train.csv")
OUTPUT_DIR_SINGLE_FILES = os.path.join(config.BASE_DIR, "data/porto_processed/")
OUTPUT_DIR_NPY = os.path.join(config.BASE_DIR, "data/porto_interpolated/")
file_interpolated = os.path.join(OUTPUT_DIR_NPY, "porto.npy")
file_normalised = os.path.join(OUTPUT_DIR_NPY, "porto_normalised.npy")

# Load raw data
df = pd.read_csv(PORTO_RAW_FILEPATH)
df.tail()

In [ ]:
# Count the number of rows with missing values (i.e., MISSING_DATA = True)
print("Number of rows with missing values:\t", df['MISSING_DATA'].sum())
# Drop rows with missing values
df = df[~df['MISSING_DATA']]
# Print number of trajectories
print("Number of trajectories:\t\t\t\t", len(df), f"[{df['TRIP_ID'].nunique()} unique]")
# If there are two trajectories with the same Trip ID, we all but the last occurrence.
# df = df.sort_values(by=['TRIP_ID', 'TIMESTAMP'])
print("Number of Duplicate Trip IDs:\t\t", df.duplicated(subset=['TRIP_ID']).sum())
df[df.duplicated(subset=['TRIP_ID'])].sort_values(by=['TRIP_ID', 'TIMESTAMP'])
# Drop those duplicates
df = df.drop_duplicates(subset=['TRIP_ID'], keep='last')
assert df['TRIP_ID'].nunique() == len(df)
print("Number of trajectories:\t\t\t\t", len(df))
# Sort by trip id for reproducibility
df = df.sort_values(by=['TRIP_ID'])

In [ ]:
# Example how to derive the trajectory for one TID
# Convert polyline to numpy array
TID = 1372636854620000520
polyline = df[df['TRIP_ID'] == TID]['POLYLINE']
# Polyline is string containing [[lon, lat], [lon, lat], ...]
# array = np.array(eval(polyline.values))
traj = np.array(eval(polyline.values[0]))
print(traj[:10])

## Extract Trajectories

In [ ]:
# We also want to store the timestamp etc. 
def get_traj(row) -> pd.DataFrame:
    tid = row['TRIP_ID'].values[0]
    uid = row['TAXI_ID'].values[0]
    traj = np.array(eval(row['POLYLINE'].values[0]))
    # The timestamp is a Unix Timestamp (in seconds). Convert to datetime. I am not 100% sure about the timezone, but I assume that this configuration is correct as all times are within the correct range. (see https://www.kaggle.com/datasets/crailtap/taxi-trajectory/data)
    unix_timestamp = row['TIMESTAMP'].values[0]
    timestamp = pd.to_datetime(unix_timestamp, unit='s', origin='unix')
    # timestamp2 = datetime.utcfromtimestamp(unix_timestamp)
    # assert str(timestamp) == str(timestamp2), f"Expected {timestamp}, got {timestamp2}."
    # Create a pandas dataframe with the trajectory:
    # - TID and UID are the same for all rows
    # - Timestamp increases by 15 seconds for each row
    dataframe = pd.DataFrame(traj, columns=['LON', 'LAT']) if len(traj) > 0 else pd.DataFrame(columns=['LON', 'LAT'])
    dataframe['TID'] = tid
    dataframe['UID'] = uid
    dataframe['TIMESTAMP'] = pd.date_range(start=timestamp, periods=len(dataframe), freq='15s')
    return dataframe


# Test the function
TID = 1372636854620000520
# TID = 1404157147620000079
get_traj(df[df['TRIP_ID'] == TID])

In [ ]:
# create a dict of format {trip_id: traj}
# traj_dict = {tid: get_traj(row=df[df['TRIP_ID'] == tid]) for tid in tqdm(df['TRIP_ID'].values)}
# Parallel version
def process_traj(tid):
    return tid, get_traj(row=df[df['TRIP_ID'] == tid])


with mp.Pool(mp.cpu_count()) as pool:
    results = list(tqdm(pool.imap(process_traj, df['TRIP_ID'].values), total=len(df['TRIP_ID'].values)))

traj_dict = dict(results)
# Results
printc(f"Number of trajectories:", len(traj_dict))
# Print one example to see it worked
traj_dict[TID]

## Bounding Box

We use a bounding box for nicer plotting

In [ ]:
x_lim = (-8.72, -8.50)  # Longitude
y_lim = (41.1, 41.24)  # Latitude

# Drop all empty trajectories
traj_dict = {tid: traj for tid, traj in traj_dict.items() if len(traj) > 0}
printc(f"Number of trajectories:", len(traj_dict))
printc(f"Dropped empty trajectories:", 1710579 - len(traj_dict))

# Create a point set with all points (lon, lat)
points_only = pd.concat(traj_dict.values())[['LON', 'LAT']].values
printc(f"Number of points:", len(points_only))

# Determine how many percent of the points are within these boarders
x_min, x_max = x_lim
y_min, y_max = y_lim

x_in = np.logical_and(points_only[:, 0] >= x_min, points_only[:, 0] <= x_max)
y_in = np.logical_and(points_only[:, 1] >= y_min, points_only[:, 1] <= y_max)
within = np.logical_and(x_in, y_in)

printc(f"Percentage of points within bounding box:", f"{within.mean() * 100:.2f}%")
n_points_within = within.sum()
printc(f"Number of points within bounding box:", n_points_within)
printc(f"Number of points outside bounding box:", len(points_only) - n_points_within)

In [ ]:
# Drop all points outside the bounding box
def filter_traj(item):
    tid, traj = item
    traj: pd.DataFrame
    x_in = np.logical_and(traj['LON'] >= x_min, traj['LON'] <= x_max)
    y_in = np.logical_and(traj['LAT'] >= y_min, traj['LAT'] <= y_max)
    within = np.logical_and(x_in, y_in)
    return tid, traj[within]


with mp.Pool(mp.cpu_count()) as pool:
    results = list(tqdm(pool.imap(filter_traj, traj_dict.items(), chunksize=1000), total=len(traj_dict)))

traj_dict = dict(results)

In [ ]:
# Double check that the number of points is correct
points_only = pd.concat(traj_dict.values())[['LON', 'LAT']].values
assert len(points_only) == n_points_within, f"Expected {n_points_within} points, got {len(points_only)}."

## Drop short trajectories 

In [ ]:
# Remove all trajectories with less than 10 points
min_points = 10
cleaned_dict = {k: v for k, v in traj_dict.items() if len(v) >= min_points}
n_removed = len(traj_dict) - len(cleaned_dict)
print(f"Removed {n_removed:,} trajectories with less than {min_points} points.")
print(f"Remaining trajectories: {len(traj_dict)}")

## Restrict Upper Length

In [ ]:
# Determine max length of trajectories
lengths = [len(traj) for traj in cleaned_dict.values()]
max_len = max(lengths)
print(f"Max length of trajectories: {max_len}")

In [ ]:
# Create a histogram of the lengths of the trajectories (only consider the area between 0 and 100)
plt.hist(lengths, bins=100, range=(0, 100), density=True)
plt.title("Length of Trajectories")
plt.xlabel("Length")
plt.ylabel("Density")

# Compute how many percent of data are longer than 100 and 200
longer_than_100 = sum([1 for l in lengths if l > 100])
longer_than_200 = sum([1 for l in lengths if l > 200])
print(f"Trajectories longer than 100: {longer_than_100} ({longer_than_100 / len(lengths) * 100:.2f}%)")
print(f"Trajectories longer than 200: {longer_than_200} ({longer_than_200 / len(lengths) * 100:.2f}%)")
# Compute the 95% quantile
q95 = np.quantile(lengths, 0.95)
print(f"95% quantile: {q95}")

In [ ]:
# Drop all trajectories longer than 100
max_len = 100
final_dict = {k: v for k, v in cleaned_dict.items() if len(v) <= max_len}
removed = len(cleaned_dict) - len(final_dict)
print(f"Removed {removed:,} trajectories with more than {max_len} points.")
print(f"Remaining trajectories: {len(final_dict):,}")

In [ ]:
# print random trajectory
tid = list(final_dict.keys())[0]
print(final_dict[tid])

 ## Saving in old dataset format
This format works well without framework and can be pushed to GitHub.

In [ ]:
# We want to write each trajectory to a separate file: tid.csv
os.makedirs(OUTPUT_DIR_SINGLE_FILES, exist_ok=True)


def save_traj(item):
    tid, traj = item
    traj.columns = traj.columns.str.lower()  # Rename all columns to lowercase
    traj.to_csv(os.path.join(OUTPUT_DIR_SINGLE_FILES, f"{tid}.csv"), index=False, sep=",")

# with mp.Pool(mp.cpu_count()) as pool:
#     list(tqdm(pool.imap(save_traj, final_dict.items()), total=len(final_dict)))

## DiffTraj Interpolated Dataset

In [ ]:
# Create a dataset with interpolated trajectories
os.makedirs(OUTPUT_DIR_NPY, exist_ok=True)
sorted_tids = sorted(final_dict.keys())
sorted_lonlat = [final_dict[tid][['LON', 'LAT']].values for tid in sorted_tids]
resampled_trajs = np.array([resample_trajectory(traj, length=max_len) for traj in tqdm(sorted_lonlat)])
print(f"Number of resampled trajectories:\t\t{len(resampled_trajs)}")
# Save into one file
# np.save(file_interpolated, resampled_trajs)

## Normalisation

We don't want to use std and mean of the whole dataset as done in DiffTraj for two reasons:
1. We want to normalise all values into a fixed range of [-1, 1].
2. We don't want to use dataset statistics for normalisation, as this would leak information about the dataset into the normalisation process.

In [ ]:
from privtrajgen.utils.helpers import get_ref_point, get_scaling_factor

npy_dataset = np.load(file_interpolated)
points = npy_dataset.reshape(-1, 2)
printc("Shape of points:", points.shape)
ref_point = get_ref_point(points)
printc(f"Reference point (LON, LAT):", f"{ref_point}")
sf = get_scaling_factor(points, ref_point)
printc(f"Scaling factor (LON, LAT):", f"{sf}")

In [ ]:
# Compute the reference point based on the bounding box
ref_point_bb = (np.mean(x_lim), np.mean(y_lim))
printc(f"Reference point (LON, LAT):", f"{ref_point}")
# Compute the scaling factor based on the bounding box
sf_bb = (np.ptp(x_lim) / 2., np.ptp(y_lim) / 2.)
printc(f"Scaling factor (LON, LAT):", f"{sf}")

# Check that they are the same (atol = 1e-6)
assert np.allclose(ref_point, ref_point_bb, atol=1e-6), f"Expected {ref_point_bb}, got {ref_point}."
assert np.allclose(sf, sf_bb, atol=1e-2), f"Expected {sf_bb}, got {sf}."

## Save Normalised & Interpolated Dataset

In [ ]:
npy_dataset = np.load(file_interpolated)
assert npy_dataset.shape == (privtrajgen.datasets.constants.NUM_PROCESSED_TRAJS_PORTO, 100,
                             2), f"Expected ({privtrajgen.datasets.constants.NUM_PROCESSED_TRAJS_PORTO}, 100, 2), got {npy_dataset.shape}."
# Normalise the trajectories
normalised_trajs = (npy_dataset - ref_point) / sf
assert normalised_trajs.min() >= -1 and normalised_trajs.max() <= 1, f"Expected [-1, 1], got [{normalised_trajs.min()}, {normalised_trajs.max()}]."
# For DiffTraj, we need to transpose, as the expected format is (B, 2, 100)
normalised_trajs = normalised_trajs.transpose(0, 2, 1)
# Save into one file
np.save(file_normalised, normalised_trajs)

## Custom Dataset Class

Test our custom dataset

In [ ]:
from privtrajgen.datasets import get_dataset

dataset = get_dataset('porto', latlon_only=True, return_labels=False, keep_original=False, normalize=False)

assert len(dataset) == 1559209, f"Expected {1559209} trajectories, got {len(dataset)}."

In [ ]:
# Load the npy file
npy_dataset = np.load(file_interpolated)

assert len(
    npy_dataset) == privtrajgen.datasets.constants.NUM_PROCESSED_TRAJS_PORTO, f"Expected {privtrajgen.datasets.constants.NUM_PROCESSED_TRAJS_PORTO} trajectories, got {len(npy_dataset)}."

In [ ]:
printc("Number of trajectories [Dataset}:", len(dataset))
printc("Number of trajectories [Numpy]:", len(npy_dataset))
printc("Length of first trajectory [Dataset]:", len(dataset[0]))
printc("Length of first trajectory [Numpy]:", len(npy_dataset[0]))
printc("Shape of first trajectory [Dataset]:", dataset[0].shape)
printc("Shape of first trajectory [Numpy]:", npy_dataset[0].shape)


### Plot Dataset

In [ ]:
# Only create points-only if undefined
if 'points_only' not in locals():
    points_only = npy_dataset.reshape(-1, 2)
# Plot points_only
NUM_POINTS = 100000
# Select points randomly to not have bias
np.random.seed(42)
indices = np.random.choice(len(points_only), NUM_POINTS, replace=False)
# Plot with scatter size 0.1
plt.figure(figsize=(10, 10))
plt.scatter(points_only[indices, 0], points_only[indices, 1], s=0.1)
plt.title("Porto Taxi Dataset")
# Restrict axis
plt.xlim(-8.72, -8.50)
plt.ylim(41.1, 41.25)
plt.xlabel("Longitude")
plt.ylabel("Latitude")

In [ ]:
# Repeat bus use the dataset class
NUM_TRAJ = NUM_POINTS // 50
# Select random trajectories
indices = np.random.choice(len(dataset), NUM_TRAJ, replace=False)
points = np.concatenate([dataset[i] for i in indices])
printc("Shape of points:", points.shape)
# Plot the selected trajectories
plt.figure(figsize=(10, 10))
# Plot with scatter size 0.1
plt.figure(figsize=(10, 10))
plt.scatter(points[:, 0], points[:, 1], s=0.1)
plt.title("Porto Taxi Dataset")
# Restrict axis
plt.xlim(-8.72, -8.50)
plt.ylim(41.1, 41.25)
plt.xlabel("Longitude")
plt.ylabel("Latitude")

In [ ]:
# Plot the normalised points
printc("Shape of points:", points.shape)
# Make dataframe, first column is lon, second is lat
points_df = pd.DataFrame(points, columns=SPATIAL_COLUMNS)
normalised_points, _, _ = normalize_points(points_df, ref=REF_POINT_PORTO, scale_factor=SCALE_FACTOR_PORTO,
                                           columns=SPATIAL_COLUMNS)

In [ ]:
# Select random normalised points
printc("Shape of points:", normalised_points.shape)
normalised_points = normalised_points.values
# Plot with scatter size 0.1
plt.figure(figsize=(10, 10))
plt.scatter(normalised_points[:, 0], normalised_points[:, 1], s=0.1)
plt.title("Porto Taxi Dataset")
# Restrict axis
# plt.xlim(-1)
# plt.ylim(41.1, 41.25)
plt.xlabel("Longitude")
plt.ylabel("Latitude")

In [ ]:
# Load the normalised npy file
npy_normalised = np.load(file_normalised)
printc("Shape of npy normalised:", npy_normalised.shape)
normalised_points = npy_normalised.transpose(0, 2, 1).reshape(-1, 2)
indices = np.random.choice(len(normalised_points), NUM_POINTS, replace=False)
# Plot the selected trajectories
plt.figure(figsize=(10, 10))
# Plot with scatter size 0.1
plt.figure(figsize=(10, 10))
plt.scatter(normalised_points[indices, 0], normalised_points[indices, 1], s=0.1)
plt.title("Porto Taxi Dataset")
# Restrict axis
# plt.xlim(-1)
# plt.ylim(41.1, 41.25)
plt.xlabel("Longitude")
plt.ylabel("Latitude")

In [ ]:
import torch
# Try the normalised dataset in a dataloader
from torch.utils.data import DataLoader

printc("Shape of normalised dataset:", npy_normalised.shape)
# Move dataset to GPU 0
npy_normalised_tensor = torch.from_numpy(npy_normalised)
dl = DataLoader(npy_normalised_tensor, batch_size=32, shuffle=True, num_workers=8, drop_last=True, pin_memory=True)
batch = next(iter(dl))
printc("Shape of batch:", batch.shape)

## Dataset To NPY

In [ ]:
from privtrajgen.datasets.difftraj_dataset import dataset_to_difftraj_npy, difftraj_conditional_paths
from privtrajgen.datasets import get_dataset

# Save the dataset as npy and see if the results are the same
tmp_file = os.path.join("/tmp", "porto_tmp.npy")

# Convert the dataset to npy
dataset = get_dataset('porto', latlon_only=True, return_labels=False, keep_original=False, normalize=True,
                      return_conditional=True)
dataset_to_difftraj_npy(dataset, tmp_file, cond_path=difftraj_conditional_paths['porto'])

In [ ]:
# Compare
npy1 = np.load(file_normalised)
npy2 = np.load(tmp_file)
printc("Results are the same:", np.allclose(npy1, npy2, atol=1e-6))

In [ ]:
# Load the conditional npy file
npy_cond = np.load(difftraj_conditional_paths['porto'])
printc("Shape of npy conditional:", npy_cond.shape)
printc("First Element of Conditional:", npy_cond[0])